In [1]:
# IGNORE THIS CODE
import os
os.environ["WANDB_DISABLED"] = "true"

### **Fine-tuning for Multi-Class Classification**

Let's put all this together in a detailed, non-trivial example. We will perform transfer learning by fine-tuning a `distilbert/distilbert-base-uncased` model on the **AG News** dataset, which involves classifying news articles into one of four categories: World (0), Sports (1), Business (2), and Sci/Tech (3).

PS. Using Google Colab's `generate` capability, I have added comments to each line of code.



#### **Step 1: Setup and Environment**

First, ensure you have the necessary libraries installed.


In [2]:
# 'accelerate' is required for the Trainer to work efficiently with GPU
! pip install transformers datasets evaluate accelerate -U

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 74.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 18.2 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.13.1
    Uninstalling transformers-5.13.1:
      Successfully uninstalled transformers-5.13.1


You'll also need to be logged into your Hugging Face account to push the model to the Hub (optional but good practice).

In [3]:
from huggingface_hub import notebook_login

notebook_login()

#### **Step 2: Load and Prepare the Dataset**

We will use the `datasets` library to load the AG News dataset and a tokenizer to prepare the text data.

In [16]:
from datasets import load_dataset
from transformers import AutoTokenizer

# Load the AG News dataset
dataset = load_dataset("fancyzhx/ag_news")

# Load the tokenizer for our chosen pre-trained model
model_checkpoint = "distilbert/distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# The AG News dataset has a 'text' column. We need a function to tokenize this text for every row in the dataset.
def tokenize_function(row):
    # The tokenizer will convert text into the input IDs and attention masks the model expects.
    # `truncation=True` ensures that inputs longer than the model's max length are cut short.
    # `padding="max_length"` can be used, but dynamic padding with a data collator is more efficient.
    return tokenizer(row["text"], truncation=True, max_length=512)

In [17]:
# Apply the tokenization function to the entire dataset.
# The `batched=True` argument processes multiple rows at once for speed.
tokenized_datasets = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [20]:
help(dataset.map)

Help on method map in module datasets.dataset_dict:

map(function: Optional[Callable] = None, with_indices: bool = False, with_rank: bool = False, with_split: bool = False, input_columns: Union[str, list[str], NoneType] = None, batched: bool = False, batch_size: Optional[int] = 1000, drop_last_batch: bool = False, remove_columns: Union[str, list[str], NoneType] = None, keep_in_memory: bool = False, load_from_cache_file: Optional[bool] = None, cache_file_names: Optional[dict[str, Optional[str]]] = None, writer_batch_size: Optional[int] = 1000, features: Optional[datasets.features.features.Features] = None, disable_nullable: bool = False, fn_kwargs: Optional[dict] = None, num_proc: Optional[int] = None, desc: Optional[str] = None, try_original_type: Optional[bool] = True, on_mixed_types: Optional[Literal['use_json']] = None) -> 'DatasetDict' method of datasets.dataset_dict.DatasetDict instance
    Apply a function to all the examples in the table (individually or in batches) and update t

In [18]:
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 7600
    })
})

In [4]:
# The 'text' columns are no longer needed in this format.
tokenized_datasets = tokenized_datasets.remove_columns(["text"])
# We also rename 'label' to 'labels' as this is the key the model expects for the labels.
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")

# Create smaller subsets for quicker training and testing - use these if you are running on local
small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(1000))
small_eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(100))

README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 18.6MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.23MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [19]:
from transformers import DataCollatorWithPadding

# Create a data collator. This will dynamically pad the batched data to the longest sentnece in each batch.
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

#### **Step 3: Load the Pre-trained Model**

We load the `distilbert-base-uncased` model using `AutoModelForSequenceClassification`. Crucially, we specify the number of labels in our dataset. This replaces the pre-trained classification head (which was for a different task) with a new, randomly initialized head suited for our 4-class problem.

In [6]:
from transformers import AutoModelForSequenceClassification

# We need to tell the model how many classes to predict. We can get this from the dataset's features.
num_labels = dataset["train"].features["label"].num_classes

model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=num_labels)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


At this point, the entire model is pre-trained except for the final classification layer. The weights of this new layer are random, and our goal during fine-tuning is to train this layer to become an effective AG News classifier.

#### **Step 4: Define Evaluation Metrics**

The `Trainer` can compute the loss during evaluation, but to get more interpretable metrics like accuracy and F1-score, we must define a `compute_metrics` function.

In [7]:
import numpy as np
import evaluate

# Load the accuracy metric from the 'evaluate' library
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    # `eval_pred` is a tuple containing the model's predictions (logits) and the true labels.
    logits, labels = eval_pred

    # The logits are the raw, unnormalized scores. We take the argmax to get the predicted class index.
    predictions = np.argmax(logits, axis=-1)

    # The metric object's compute method returns a dictionary with the metric name and value.
    return metric.compute(predictions=predictions, references=labels)

#### **Step 5: Configure `TrainingArguments`**

Now, we create an instance of `TrainingArguments` to define our comprehensive training strategy.

In [8]:
from transformers import TrainingArguments

# Define a name for our new model on the Hugging Face Hub
model_name = "trainingone"

training_args = TrainingArguments(
    output_dir=model_name,
    # --- Performance and Resource Management ---
    learning_rate=2e-5, # 2/100000 = 0.00002
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=True,  # Use mixed precision if a compatible GPU is available

    # --- Logging, Saving, and Evaluation Strategy ---
    eval_strategy="epoch",       # Evaluate at the end of each epoch
    save_strategy="epoch",       # Save a checkpoint at the end of each epoch
    logging_strategy="epoch",    # Log metrics at the end of each epoch

    # --- Model Loading and Hub Integration ---
    load_best_model_at_end=True, # Load the best performing model checkpoint at the end
    metric_for_best_model="accuracy", # Use accuracy to determine the best model
    push_to_hub=True, # Upload the final model to the Hub
)

This configuration sets up a robust training run that evaluates and saves the model every epoch, tracks the best model based on accuracy, uses mixed-precision for speed, and shares the result.

#### **Step 6: Instantiate and Run the `Trainer`**

With all the components ready, we can now instantiate the `Trainer` and start the fine-tuning process.

In [9]:
from transformers import Trainer

trainer = Trainer(
    model=model, # pretrained model
    args=training_args, # training arguments defined earlier
    train_dataset=tokenized_datasets["train"], # training dataset
    eval_dataset=tokenized_datasets["test"], # evaluation dataset
    compute_metrics=compute_metrics, # evaluation metrics
    data_collator=data_collator, # dynamic padding
)

# Start the training process!
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.228802,0.175914,0.941842
2,0.138894,0.181763,0.948816
3,0.093250,0.218364,0.946184


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=22500, training_loss=0.1536489963107639, metrics={'train_runtime': 1292.4108, 'train_samples_per_second': 278.549, 'train_steps_per_second': 17.409, 'total_flos': 9056469134955648.0, 'train_loss': 0.1536489963107639, 'epoch': 3.0})

When you run `trainer.train()`, you will see a progress bar and a log of the training process, which will look something like this after each epoch:

```
Epoch | Training Loss | Validation Loss | Accuracy
--------------------------------------------------
  1   |     0.6582    |      0.3512     |  0.8850
  2   |     0.2845    |      0.2987     |  0.9010
  3   |     0.1879    |      0.2955     |  0.9070

TrainOutput(global_step=189, training_loss=0.3712, metrics={'...'})
```

This output clearly shows the model's performance improving over time. Because we set `load_best_model_at_end=True` and `metric_for_best_model="accuracy"`, the `trainer` object now holds the weights from the end of Epoch 3, which achieved the highest accuracy of `0.9070`.



#### **Step 7: Push to Hub and Make a Prediction**

The final step is to share your model and use it for inference.

In [10]:
# The trainer.push_to_hub() command will upload your model, tokenizer,
# and training configuration to the Hub under the name you specified.
trainer.push_to_hub()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

CommitInfo(commit_url='https://huggingface.co/sampurn-gfg/trainingone/commit/a4091cd5c728fc5cc2b25e65ad32c1f8c4036de4', commit_message='End of training', commit_description='', oid='a4091cd5c728fc5cc2b25e65ad32c1f8c4036de4', pr_url=None, repo_url=RepoUrl('https://huggingface.co/sampurn-gfg/trainingone', endpoint='https://huggingface.co', repo_type='model', repo_id='sampurn-gfg/trainingone'), pr_revision=None, pr_num=None)

In [22]:
from transformers import pipeline
from datasets import load_dataset

model_name = "sampurn-gfg/trainingone"
dataset = load_dataset("fancyzhx/ag_news")

# A new sports-related text
text = "Hugging Face Trainer API allows for smooth transfer learning code"
# text = "The business operations ran smoothly."

# Create a pipeline with our fine-tuned model
classifier = pipeline("sentiment-analysis", model=model_name)

# Make a prediction
prediction = classifier(text)
print(prediction)

# We can also check the label mapping
print(dataset["train"].features["label"].names)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'LABEL_3', 'score': 0.9956005811691284}]
['World', 'Sports', 'Business', 'Sci/Tech']


The output would be:

```
[{'label': 'LABEL_1', 'score': 0.905}]
['World', 'Sports', 'Business', 'Sci/Tech']
```

The model correctly predicts `LABEL_1` with high confidence, which corresponds to the "Sports" category. This demonstrates that our transfer learning process was successful.

# How to create your own Dataset

In [12]:
!pip install datasets

In [13]:
from datasets import Dataset, Features, Image, Value
import pandas as pd

# Text Dataset
text_data = {
    "text": ["Hello world", "Hugging Face is great", "Machine learning is fun"],
    "label": [1, 1, 0]
}
text_dataset = Dataset.from_dict(text_data)
print("Text Dataset:", text_dataset)

# Image Dataset
# Assuming you have images in a folder; here we define the structure
# Note: To run this, you'd need actual image files at these paths
image_data = {
    "image": ["path/to/image1.jpg", "path/to/image2.png"],
    "label": [0, 1]
}
# Define features to tell the library the 'image' column contains images
features = Features({"image": Image(), "label": Value("int64")})
image_dataset = Dataset.from_dict(image_data, features=features)

# PDF (Text Extraction) Dataset
# To handle PDFs, you typically use a library like 'pypdf' or 'pdfplumber'
# !pip install pypdf

# Example of how you would structure the extracted content:
pdf_data = {
    "pdf_name": ["report.pdf"],
    "content": ["This is the extracted text from the PDF..."],
    "metadata": ["Author: John Doe"]
}
pdf_dataset = Dataset.from_dict(pdf_data)

display(text_dataset[0])

Text Dataset: Dataset({
    features: ['text', 'label'],
    num_rows: 3
})


{'text': 'Hello world', 'label': 1}

In [14]:
from transformers import TrainingArguments
help(TrainingArguments)

Help on class TrainingArguments in module transformers.training_args:

class TrainingArguments(builtins.object)
 |  TrainingArguments(output_dir: str | None = None, per_device_train_batch_size: int = 8, num_train_epochs: float = 3.0, max_steps: int = -1, learning_rate: float = 5e-05, lr_scheduler_type: transformers.trainer_utils.SchedulerType | str = 'linear', lr_scheduler_kwargs: dict | str | None = None, warmup_steps: float = 0, optim: transformers.training_args.OptimizerNames | str = 'adamw_torch_fused', optim_args: str | None = None, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, optim_target_modules: None | str | list[str] = None, gradient_accumulation_steps: int = 1, average_tokens_across_devices: bool = True, max_grad_norm: float = 1.0, label_smoothing_factor: float = 0.0, bf16: bool = False, fp16: bool = False, bf16_full_eval: bool = False, fp16_full_eval: bool = False, tf32: bool | None = None, gradient_checkpointing